# LeadCraft (v0.1): Activity Retrieval, Caching & Similarity Search

This notebook demonstrates how to use and what to expect from the project.

## Overview

The system performs the following operations:

1. **Target Search**: Searches ChEMBL database for biological targets if not cached
2. **Activity Retrieval**: Fetches/calculates bioactivity data, assay content, molecular structures
3. **Data Processing**: Calculates pChEMBL values and classifies activity, deduplicate measurements (activity-aware operation)
4. **Caching**: Stores data in SQLite for fast subsequent queries/reuse.
5. **Similarity Search**: Finds molecules similar to your query using Tanimoto similarity (MACCS keys)

**Outputs**
- Filtered DataFrame of similar, bioactive molecules.
- Database with cached target and activity info

⚠ Be aware: first run may take longer (5-60 minutes depending on the target) due to remote ChEMBL calls.


___

### Setup and Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.mol_activity.utils.logging_utils import setup_logging
from src.mol_activity.main import basic_query

# Configure logging
setup_logging()

# Set pandas display options (to avoid column/row collapsing)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

---

### Example: CDK6 Kinase Inhibitor Search

In this example, we'll search for compounds similar to a known CDK6 inhibitor.

**Query Molecule**: Abemaciclib (a known CDK4/6 inhibitor)
**SMILES**: `CCN1CCN(CC1)Cc2ccc(nc2)Nc3ncc(c(n3)c4cc5c(c(c4)F)nc(n5C(C)C)C)F`
**Target**: CDK6 (Cyclin-dependent kinase 6)

In [2]:
# Define search parameters
target_name = "CDK6"
query_smiles = "CCN1CCN(CC1)Cc2ccc(nc2)Nc3ncc(c(n3)c4cc5c(c(c4)F)nc(n5C(C)C)C)F"
min_similarity = 0.7
max_molecules = 15
organism = "Homo sapiens"

### Run the Query

The first run will fetch data from ChEMBL (may take 5-60 minutes). Subsequent runs will use cached data and return data within 1-2 minutes. Please, note that ChEMBL api could be unstable, not proceed with requests/raise errors. It keeps the progress, though. Reiteration of this cell in case of the error or long response time could be necessary.

In [3]:
results = basic_query(
    target=target_name,
    smiles=query_smiles,
    min_similarity=min_similarity,
    max_molecules=max_molecules,
    organism=organism,
)

print(
    f"\nFound {len(results)} activity records from {results['molecule_chembl_id'].nunique()} unique molecules"
)

2026-03-09 18:49:22 - INFO - src.mol_activity.main - Starting query pipeline for target: 'CDK6'
2026-03-09 18:49:22 - INFO - database.files_and_SQL_utils - Database initialized at /home/viktor/PycharmProjects/lead_craft_api/data/molecular_activities.db
2026-03-09 18:49:22 - INFO - database.files_and_SQL_utils - Target 'CDK6' not found in database
2026-03-09 18:49:22 - INFO - database.files_and_SQL_utils - Fetching fresh data from ChEMBL...
2026-03-09 18:49:22 - INFO - src.mol_activity.utils.mol_activity_data_utils - Starting main pipeline to generate activities for query: 'CDK6'

2026-03-09 18:49:22 - INFO - src.mol_activity.utils.mol_activity_data_utils - Step 1/7: Search for targets matching query
2026-03-09 18:49:23 - INFO - src.mol_activity.utils.mol_activity_data_utils - Searching for targets with query: 'CDK6', organism: 'Homo sapiens
2026-03-09 18:49:25 - INFO - src.mol_activity.utils.mol_activity_data_utils - Found 13 targets matching criteria
2026-03-09 18:49:25 - INFO - src.m


Found 25 activity records from 15 unique molecules


---

## Exploring the Results

### Preview the Data

In [4]:
results.iloc[
    11:20
]  # the fist several rows are for the same compound but different (sub)targets/assays/context

,molecule_chembl_id,activity_id,target_chembl_id,assay_chembl_id,pchembl_value,context,canonical_smiles,is_active,tanimoto_similarity
11,CHEMBL5589797,25993923,CHEMBL2111455,CHEMBL5551301,7.11000,biochemical,CCN1CCN(Cc2ccc(Nc3ncc(F)c(N4CCc5ncn(C(C)C)c5C4...,True,1.000000
12,CHEMBL5804149,27792164,CHEMBL2508,CHEMBL5734561,7.22000,biochemical,CCN1CCN(Cc2ccc(Nc3ncc(F)c(-c4ncc5nc(C)n(C(C)C)...,True,1.000000
13,CHEMBL6013263,27792398,CHEMBL2508,CHEMBL5734561,6.52000,biochemical,CCN1CCN(c2ccc(Nc3ncc(F)c(-c4ncc5nc(C)n(C(C)C)c...,True,1.000000
14,CHEMBL6026269,27792116,CHEMBL2508,CHEMBL5734561,7.69897,biochemical,CCN1CCN(Cc2ccc(Nc3ncc(F)c(-c4ccc5c(ccn5C(C)C)c...,True,1.000000
15,CHEMBL4203179,18538794,CHEMBL2111448,CHEMBL4197054,8.80000,biochemical,CCN1CCC(N2CCc3nc(Nc4ncc(F)c(-c5cc(F)c6nc(C)n(C...,True,0.982143
16,CHEMBL5087355,24415157,CHEMBL2111455,CHEMBL5047192,7.44000,biochemical,CCC(C)n1cc(-c2nc(Nc3ccc(CN4CCN(C(C)C)CC4)cn3)n...,True,0.982143
17,CHEMBL5084067,24415148,CHEMBL2111455,CHEMBL5047192,8.00000,biochemical,CC(C)N1CCN(Cc2ccc(Nc3ncc(F)c(-c4cn(C(C)C)c5ncc...,True,0.981818
18,CHEMBL4211828,28006019,CHEMBL2111448,CHEMBL5735351,4.90000,biochemical,CC(C)N1CCN(Cc2ccc(Nc3ncc(F)c(-c4cc(F)c5nc6n(c5...,False,0.964286
19,CHEMBL4211828,18497348,CHEMBL2508,CHEMBL4188522,7.90000,cellular,CC(C)N1CCN(Cc2ccc(Nc3ncc(F)c(-c4cc(F)c5nc6n(c5...,True,0.964286


### Column Descriptions

- **activity_id**: Unique ChEMBL activity identifier
- **molecule_chembl_id**: ChEMBL molecule identifier
- **target_chembl_id**: ChEMBL target identifier
- **assay_chembl_id**: ChEMBL assay identifier
- **pchembl_value**: -log10(Molar) activity value (higher = more potent)
- **context**: Assay type (biochemical, cellular, organism)
- **canonical_smiles**: Standardized SMILES representation
- **is_active**: Boolean classification based on pChEMBL threshold
- **tanimoto_similarity**: Similarity to query molecule (0-1)

### Summary Statistics

In [5]:
print("Summary Statistics")
print("=" * 60)
print(f"Total activity records: {len(results)}")
print(f"Unique molecules: {results['molecule_chembl_id'].nunique()}")
print(f"Unique targets: {results['target_chembl_id'].nunique()}")
print(f"Unique assays: {results['assay_chembl_id'].nunique()}")
print(
    f"\nSimilarity range: {results['tanimoto_similarity'].min():.3f} - {results['tanimoto_similarity'].max():.3f}"
)
print(
    f"pChEMBL range: {results['pchembl_value'].min():.2f} - {results['pchembl_value'].max():.2f}"
)

Summary Statistics
Total activity records: 25
Unique molecules: 15
Unique targets: 4
Unique assays: 13

Similarity range: 0.964 - 1.000
pChEMBL range: 3.47 - 9.37


---

## Database Management

You can also interact directly with the database for more advanced operations.

In [6]:
from src.config import settings
from database.files_and_SQL_utils import MolecularActivityDatabase

db = MolecularActivityDatabase(
    str(settings.db_path),
)  # database is stored at lead_craft_api/data/molecular_activities.db

# Get database statistics
stats = db.get_database_stats()
print("Database Statistics")
print("=" * 40)
print(f"Targets cached: {stats['targets']}")
print(f"Subtargets: {stats['subtargets']}")
print(f"Total activities: {stats['activities']}")

# List all cached targets
print("\nCached Targets:")
print("-" * 40)
targets = db.list_all_targets()
for target in targets:
    print(
        f"{target['target_name']:20s} | Subtargets: {target['num_subtargets']:2d} | Last updated: {target['created_or_updated_at']}"
    )

2026-03-09 18:53:11 - INFO - database.files_and_SQL_utils - Database initialized at /home/viktor/PycharmProjects/lead_craft_api/data/molecular_activities.db


Database Statistics
Targets cached: 1
Subtargets: 8
Total activities: 2308

Cached Targets:
----------------------------------------
CDK6                 | Subtargets:  8 | Last updated: 2026-03-09 17:53:07
